# Hongkong Horse Racing – Exploratory Data Analysis

This notebook provides an exploratory data analysis (EDA) of the Hongkong Horse Jockey Club dataset.  
It covers an initial overview, data quality checks, feature inspection, and basic preprocessing steps.  
The goal is to understand the structure, content, and potential issues of the dataset before moving on to feature engineering and modeling.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
df = pd.read_csv("../data/HKHJC_FINAL.csv")

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dtypes

It can be seen that the column types are not entirely right yet. Next, we will take a closer look at the observations.

In [ ]:
df.head(10)

In [ ]:
df.isna().sum()

We see that we have a big number of missing values in the sec-columns. This is expected, as only longer runs get more than 3 sectional times. A sectional time is the time a horse needed between two sections. Logically, if the race track gets longer, more sections are included.

Next, we take a closer look at the missing values in the horse_number column.

In [ ]:
df[df["horse_number"].isnull()][['date', 'race_no', 'horse_name', 'place', 'horse_number']]

It can be seen that every time the horse number contains a missing value, the place is marked as "WV", which stands for "withdrawal". This means the horse could not take part in the race, mostly because of short-term issues like injuries.

In [ ]:
df_copy = df.copy()

## Cleaning and Preprocessing the Columns

### race_no

First, we are going to split up the column "race_no", as it contains two separate pieces of information: the race id (seasonal id), and the race number of the day. During a season typically around 800 races take place. There are different race days, and on each day there are around 10-12 races.

In [ ]:
df['race_id_number'] = df['race_no'].str.split().str[1].astype(int) #Defines the Race Number on that given day (usually 1-12)
df['race_id_year'] = df['race_no'].str.extract(r"\((\d+)\)").astype(int) #defines the race number in that given season (usually between 1 and ~800)

### date
Next, we will transform the date into datetime format.

In [ ]:
df['date'] = pd.to_datetime(df['date'], dayfirst=True) #Converts date to datetime object
df['date'] = df['date'].dt.tz_localize('Asia/Hong_kong')

### info
The info column currently contains three pieces of information:

1. Class: Indicates the quality or level of the race, with lower numbers (e.g. Class 1) representing higher-quality races, with higher-quality horses.
2. Distance: Specifies the length of the race, measured in meters (e.g. 1200M is a sprint)
3. Rating Range: Defines the range of official handicap ratings (e.g. 60–40) that horses must fall within to be eligible for the race.

we will now split these three ratings into three own columns.

Class 5 - 1400M - (40-20)

In [ ]:
df['info_class'] = df['info'].str.split('-').str[0].str.strip()
df['info_distance'] = df['info'].str.split('-').str[1].str.strip().str.replace("M", "").astype(int)
df['info_rating'] = df['info'].str.split('-').str[2].str.strip().str.replace("(", "") + "-" + df['info'].str.split('-').str[3].str.strip().str.replace(")", "")

### going
Next we will take a look at the variable "going".
Going refers to the condition of the racetrack surface, which can significantly affect a horse’s performance.

In [ ]:
df['going'].unique()

In [ ]:
df['going'].value_counts()

For now this looks good. Later, when preprocessing for the ML model, we will treat this column further.
"SEALED" refers to a special kind of racing track: This track is designed to keep moisture out before the race. This value potentially needs special treatment.

### course
This variable contains information about the race track
1. Surface Type: TURF (grass) or DIRT (dirt)
2. Course: This refers to the rail position, i.e., how far the inside rail is moved away from the true inside of the turf track.
    e.g. "A+3": Rail moved out 3 meters from the A position

Firstly, we will only split the column into two, and later decide how we will preprocess them.

In [ ]:
df['course'].unique()

In [ ]:
def extract_surface(course_str):
    if pd.isna(course_str):
        return 'UNKNOWN'
    elif course_str == 'ALL WEATHER TRACK':
        return 'AWT'
    else:
        return 'TURF'

In [ ]:
def rail_offset(course_str):
    try:
        val = course_str.split('-')[1].split('"')[1]
        if pd.isna(course_str):
            return 0
        elif '+' in val:
            return int(val.split('+')[1])
        elif course_str == 'ALL WEATHER TRACK':
            return 0
        elif any(letter in val for letter in ['A', 'B', 'C']):
            return 0
        else:
            return 'UNKNOWN'
    except:
        return 0

In [ ]:
df['surface_type'] = df['course'].apply(extract_surface)
df['rail_offset'] = df['course'].apply(rail_offset)
#df['course_prep'] = df['course'].str.split('-').str[1].str.split('"').str[1]

In [ ]:
df['surface_type'].unique()

In [ ]:
df['rail_offset'].value_counts(dropna=False)

In [ ]:
df.dtypes

### Sectional Times
Here we will treat the sectional times

In [ ]:
print(df.columns.get_loc('sec5'))
print(df.columns.get_loc('sec6'))

In [ ]:
cols = list(df.columns)

sec_6 = cols.pop(23)
cols.insert(10, sec_6)
sec_5 = cols.pop(23)
cols.insert(10, sec_5)

df = df[cols]

In [ ]:
df.head()

In [ ]:
def replace_time(time_str):
    if pd.isna(time_str):
        return np.nan
    else:
        return time_str.replace('(', "").replace(')', "")

In [ ]:
df['sec1'] = df['sec1'].apply(replace_time)
df['sec2'] = df['sec2'].apply(replace_time)
df['sec3'] = df['sec3'].apply(replace_time)
df['sec4'] = df['sec4'].apply(replace_time)
df['sec5'] = df['sec5'].apply(replace_time)
df['sec6'] = df['sec6'].apply(replace_time)

Next we will convert the times to float, so we can later engineer fitting features for the ML model.

In [ ]:
def time_to_seconds(times):
    try:
        if pd.isna(times):
            return np.nan
        t = str(times)
        if ':' in t:
            minutes, seconds = t.split(':')
            return int(minutes) * 60 + float(seconds)
        else:
            return float(t)
    except:
        return np.nan

In [ ]:
df['sec1'] = df['sec1'].apply(time_to_seconds)
df['sec2'] = df['sec2'].apply(time_to_seconds)
df['sec3'] = df['sec3'].apply(time_to_seconds)
df['sec4'] = df['sec4'].apply(time_to_seconds)
df['sec5'] = df['sec5'].apply(time_to_seconds)
df['sec6'] = df['sec6'].apply(time_to_seconds)

In [ ]:
df['sec5'].sort_values(ascending=False)

This looks good.

In [ ]:
df.iloc[:, 11:20].head()

### Place
This feature contains the final placement of the horse

In [ ]:
place_values = list(df['place'].value_counts(dropna=False).items())

In [ ]:
print(place_values)

We see a range of different values:

| Value     | Meaning                                |
|-----------|----------------------------------------|
| `'1'`–`'14'` | Placement in the race (1st, 2nd, etc.) |
| `'x DH'`  | Dead Heat – shared placement (e.g., 2 DH = shared 2nd) |
| `'WV'`    | Withdrawn – horse was withdrawn before the race |
| `'WV-A'`  | Withdrawn (automated or special case)   |
| `'WX'`    | Withdrawn after betting started         |
| `'WX-A'`  | Withdrawn after betting (automated or adjusted) |
| `'WXNR'`  | Withdrawn, Non-Runner – horse declared but did not run |
| `'PU'`    | Pulled Up – horse was stopped mid-race (e.g., injury) |
| `'UR'`    | Unseated Rider – jockey fell off during race |
| `'DNF'`   | Did Not Finish – horse did not complete the race |
| `'DISQ'`  | Disqualified – horse was disqualified from the race |
| `'FE'`    | Fell – horse fell during the race        |
| `'TNP'`   | Took No Part – horse officially entered but did not participate |

As the model should later predict the placement of the horse, all non-numeric values are useless. However, they are important for later feature engineering, which is why we will keep them (for now). We will first create a new column "numeric_placement", where we store the placement as float, and assign NaN to the different cases.

In [ ]:
def placement(val):
    numb_list = [str(i) for i in range(1,15)]
    if val in numb_list:
        return int(val)
    elif 'DH' in val:
        return int(val.split(' ')[0])
    else:
        return np.nan

In [ ]:
df['place_numeric'] = df['place'].apply(placement)

### horse_number

Next, we will take a look at the horse number.

In [ ]:
df['horse_number'].value_counts(dropna=False)

The horse number is already correctly classified.

### horse_name
The horse name is important for identifying the horse. We will take a look at the nature of this feature.

In [ ]:
len(df['horse_name'].unique())

In [ ]:
list(df['horse_name'].unique())

In [ ]:
df['horse_name'].value_counts().describe()

### jockey
Lets do the same thing with the jockey column

In [ ]:
len(df['jockey'].unique())

In [ ]:
df['jockey'].value_counts().describe()

In [ ]:
df['jockey'].value_counts()

This statistic is interesting. We will explore it further to a later point.

### trainer
Now lets look at the trainers

In [ ]:
len(df['trainer'].unique())

In [ ]:
df['trainer'].value_counts().describe()

In [ ]:
df['trainer'].value_counts()

This statistic is also very interesting. We will come back to it at a later point.

### weight
We now take a look at the weight of the horses

In [ ]:
df['weight'].dtype

In [ ]:
df['weight'].describe()

This statistical summary makes sense. No need to change anything here.

### on_date_horse_weight
This feature describes the weight on the race date

In [ ]:
df['on_date_horse_weight'] = df_copy['on_date_horse_weight']

In [ ]:
wrong_list = []
for value in df['on_date_horse_weight']:
    try:
        test = int(value)
    except:
        wrong_list.append(value)
print(wrong_list)

There are some missing values here. Lets explore them further.

In [ ]:
def convert_values(value):
    try:
        return int(value)
    except:
        return np.nan

In [ ]:
df['on_date_horse_weight'] = df['on_date_horse_weight'].apply(convert_values)

In [ ]:
df['on_date_horse_weight'].describe()

In [ ]:
df[df['on_date_horse_weight'].isna()].head()

In [ ]:
df[df['on_date_horse_weight'].isna()]['place'].unique()

This explains it. Every time an observation does not contain a weight, the horse was withdrawn from the race. As these observations will be deleted anyway, we do not have to worry about them for now.

### draw
The Draw is the starting position of the horse.

In [ ]:
df['draw'].unique()

In [ ]:
df['draw'].value_counts()

In [ ]:
df['draw'] = df['draw'].apply(convert_values)

Here too the missing values are linked to the withdrawal of the respective horse.

In [ ]:
df.iloc[:, 20:30].head()

### length_behind_winner
This feature describes how close this observation was to the winning horse.

In [ ]:
df['length_behind_winner'].unique()

In [ ]:
print(df['length_behind_winner'].value_counts().to_string())

In [ ]:
df[df['length_behind_winner'] == '+NOSE']

In [ ]:
df[df['length_behind_winner'] == '+N']

These values are probably mistakes. We will assign them the right values.

In [ ]:
df.loc[df['length_behind_winner'] == '+NOSE', 'length_behind_winner'] = 'NOSE'
df.loc[df['length_behind_winner'] == '+N', 'length_behind_winner'] = 'N'
df.loc[df['length_behind_winner'] == '+1/2', 'length_behind_winner'] = '1/2'

In [ ]:
def is_int(t):
    try:
        int(t)
        return True
    except ValueError:
        return False

In [ ]:
def lbw_func(value):
    if value == '---':
        return np.nan
    elif value == '-':
        return 0
    elif '-' in value:
        split = value.split('-')
        return int(split[0]) + float(split[1].split('/')[0]) / float(split[1].split('/')[1])
    elif is_int(value):
        return int(value)
    elif '/' in value:
        return float(value.split('/')[0]) / float(value.split('/')[1])
    elif value == 'NOSE':
        return 0.01
    elif value == 'N':
        return 0.3
    elif value == 'SH':
        return 0.1
    elif value == 'HD':
        return 0.2
    else:
        return np.nan
    

In [ ]:
df['length_behind_winner'] = df['length_behind_winner'].apply(lbw_func)

In [ ]:
df['length_behind_winner'].describe()

### running_position
This feature shows the placement of the horse during each sectional measurement.

E.g. Draw: 10, Place: 7

Running Position might be 10 10 9 7 during 4 sectional times.

In [ ]:
df['running_position'].head(20)

This feature is a bit tough to untangle. We leave this area as it is.

### finish_time
This feature gets converted to seconds

In [ ]:
df['finish_time_seconds'] = df['finish_time'].apply(time_to_seconds)

### win_odds

In [ ]:
print(df['win_odds'].value_counts(dropna=False).to_string())

In [ ]:
df.loc[df['win_odds'] == '---', 'win_odds'] = np.nan

In [ ]:
df['win_odds'] = df['win_odds'].astype(float)

In [ ]:
df['win_odds'].dtype

In [ ]:
df.dtypes

In [ ]:
df.columns

In [ ]:
collection = ['horse_number', 'weight', 'on_date_horse_weight', 'draw', 'length_behind_winner', 'win_odds', 'race_id_number', 'race_id_year', 'info_distance', 'rail_offset', 'place_numeric', 'finish_time_seconds']

In [ ]:
correlation = df[collection].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', annot_kws={"size": 7})
plt.show()

In [ ]:
df.to_csv('../data/HKHJ_Dataset_Prepared.csv', index=False)